### Descrição da Etapa de Pré-processamento (Metodologia)

A etapa de pré-processamento do *corpus* textual consistiu em um pipeline de normalização e limpeza de dados visando a padronização das estruturas sintáticas e a integridade semântica dos documentos. Inicialmente, procedeu-se à remoção de ruídos textuais, especificamente *tags* HTML, por meio de expressões regulares. Em seguida, aplicou-se a padronização de espaçamentos, compreendendo a eliminação de espaços múltiplos e a supressão de espaços indevidos antes de vírgulas. De modo a preservar a morfologia de terminologias e abreviações próprias do domínio jurídico — como "art." (artigo), "c.c." (combinado com) e "n." (número) —, implementou-se uma heurística de consolidação que protege esses termos contra regras genéricas de tokenização e espaçamento de pontuação. Para tal, utilizou-se uma estratégia de substituição temporária por marcadores (*placeholders*), garantindo que a regra de inserção de espaço após pontos finais fosse aplicada exclusivamente para delimitação de sentenças, não afetando os acrônimos jurídicos mapeados nem a pontuação interna de valores numéricos.

#### Justificativa Metodológica: A Preservação de Stopwords e Estruturas Morfossintáticas

No contexto do Processamento de Linguagem Natural (PLN) tradicional, baseado em representações vetoriais esparsas como *Bag of Words* (BoW) ou TF-IDF, a remoção de *stopwords* (termos de alta frequência funcional, mas de baixo valor semântico isolado) era uma prática padrão para reduzir a dimensionalidade dos dados e mitigar ruídos.

Contudo, com o advento das arquiteturas baseadas em *Transformers*, como o BERT (*Bidirectional Encoder Representations from Transformers*), o paradigma de pré-processamento sofreu uma alteração fundamental. O poder preditivo do BERT reside justamente no seu mecanismo de **atenção bidirecional (Self-Attention)**, que calcula a representação de uma palavra com base em todo o contexto frasal em que ela se insere. Dessa forma, a remoção arbitrária de termos relacionais e funcionais prejudica a integridade sintática e, por conseguinte, a capacidade do modelo de capturar nuances linguísticas.

Nesta pesquisa, optou-se pela manutenção do texto o mais próximo possível de sua estrutura original, justificando-se a retenção das seguintes classes gramaticais, cruciais mormente na linguagem jurídica:

*   **Artigos:** Definem não apenas o gênero e número, mas a determinância do sujeito ou objeto em uma lide (e.g., a diferença entre "o autor" e "um autor"). A ausência de artigos fragmenta a identificação de entidades nomeadas (NER).
*   **Conjunções:** São as espinhas dorsais da argumentação jurídica. Conjunções adversativas ("mas", "porém") invertem a polaridade de uma sentença, enquanto as condicionais ("se", "caso") e conclusivas ("portanto") definem a relação de causa e efeito em uma decisão. Removê-las destrói a cadeia lógica de um acórdão ou petição.
*   **Advérbios:** Atuam como modificadores de intensidade, tempo e, criticamente, de negação. A supressão de um advérbio de negação ("não provido", "nunca", "jamais") resulta em uma inversão semântica completa e catastrófica para tarefas de classificação temática ou análise de sentimento jurisprudencial.
*   **Pronomes (Em especial a Colocação Pronominal):** O domínio do texto jurídico é marcado pelo uso formal da norma culta, incluindo frequentes ênclises e mesóclises ("condenou-se", "dar-se-á"). A preservação destas estruturas hifenizadas é vital. Os pronomes reflexivos e apassivadores carreiam informações essenciais sobre o agente da ação e a voz verbal (ex: voz passiva sintética). Quando mecanismos de limpeza quebram expressões como "condenou-se" em "condenou" e "se" (removendo este último como *stopword*), o modelo perde a informação sintática de que a ação recai sobre o sujeito, alterando o valor referencial da sentença. Além disso, pronomes relativos ("que", "o qual") são responsáveis por introduzir orações subordinadas, essenciais para que o BERT mapeie as dependências sintáticas de longo alcance na estrutura da frase.

Portanto, a decisão de limitar a limpeza à padronização de espaçamentos e remoção de ruídos estritamente não-textuais (como *tags* HTML) visa garantir que o tokenizador *WordPiece* do BERT receba sentenças sintaticamente ricas e íntegras. Tal abordagem garante a maximização da acurácia do modelo de linguagem na representação do complexo tecido textual que compõe a semântica jurídica.

In [ ]:
from nltk.tokenize import word_tokenize
import re
import nltk

try:
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except Exception as e:
    print(f"Aviso: falha ao baixar recursos NLTK: {e}")

# 1. Definindo a lista de stopwords base com as customizações necessárias
portuguese_stopwords_list = set(nltk.corpus.stopwords.words('portuguese'))
portuguese_stopwords_list.update(['e', 'é', 'o', 'a', 'os', 'as', 'nesta', 'neste', 'desse', 'dessa'])

# Removendo palavras críticas da lista de stopwords para preservar a semântica jurídica
portuguese_stopwords_list.discard('não')
portuguese_stopwords_list.discard('nem')
portuguese_stopwords_list.discard('sem')
portuguese_stopwords_list.discard('ou')
portuguese_stopwords_list.discard('mas')

# Configurações de Placeholders para preservação de termos
_SIMPLE_ABBREVS = ['art', 'n', 'fls', 'fl', 'inc', 'min', 'des', 'rel', 'cf', 'doc', 'proc', 'dr', 'dra']
_DIGIT_DOT_PH = "_#_DIGIT_DOT_#_"
_CC_DOT_PH = "_#_CC_DOT_#_"   # c.c.
_CC_PH = "_#_CC_#_"           # c.c
_NRO_PH = "_#_NRO_#_"         # nº

def _abbr_ph(abbr):
    return f"_#_{abbr.upper()}_DOT_#_"

def remove_noise(text):
    if isinstance(text, str):
        clean = re.compile('<.*?>')
        return re.sub(clean, ' ', text)
    return text

def _consolidate_known_acronyms(text):
    text = re.sub(r'(?i)(\bart)\s+\.', r'\1.', text)
    text = re.sub(r'(?i)(\bn)\s+\.', r'\1.', text)
    text = re.sub(r'(?i)(c)\s+\.\s*(c)\s+\.', r'\1.\2.', text)
    text = re.sub(r'(?i)(c)\s+\.\s*(c)', r'\1.\2', text)
    text = re.sub(r'(?i)(c\.)\s+(c)', r'\1\2', text)
    for abbr in _SIMPLE_ABBREVS:
        if abbr not in ('art', 'n'):
            text = re.sub(r'(?i)(\b' + re.escape(abbr) + r')\s+\.', r'\1.', text)
    return text

def _protect_dots(text):
    text = re.sub(r'(\d)\.(\d)', r'\1' + _DIGIT_DOT_PH + r'\2', text)
    text = re.sub(r'(?i)\bnº', _NRO_PH, text)
    text = re.sub(r'(?i)\bc\.c\.', _CC_DOT_PH, text)
    text = re.sub(r'(?i)\bc\.c(?!\w)', _CC_PH, text)
    for abbr in _SIMPLE_ABBREVS:
        ph = _abbr_ph(abbr)
        text = re.sub(r'(?i)(\b' + re.escape(abbr) + r')\.', r'\1' + ph, text)
    return text

def _restore_dots(text):
    text = text.replace(_DIGIT_DOT_PH, '.')
    text = text.replace(_NRO_PH, 'nº')
    text = text.replace(_CC_DOT_PH, 'c.c.')
    text = text.replace(_CC_PH, 'c.c')
    for abbr in _SIMPLE_ABBREVS:
        text = text.replace(_abbr_ph(abbr), '.')
    return text

def remove_whitespace(text):
    if not isinstance(text, str): return text
    text = re.sub(r'\s+', ' ', text).strip()
    text = _consolidate_known_acronyms(text)
    text = re.sub(r'\s+([,:;])', r'\1', text)
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)
    text = re.sub(r'§\s*', '§ ', text)

    text = _protect_dots(text)
    text = re.sub(r'\s+\.', '.', text) # Remove espaços antes de pontos
    text = re.sub(r'\.(?=\S)', '. ', text) # Garante espaço após ponto de sentença
    text = _restore_dots(text)
    return re.sub(r'\s+', ' ', text).strip()

def remove_pontuacao(texto):
    if not isinstance(texto, str): return texto
    texto = _protect_dots(texto)
    # Hífen (\-) removido da regex para manter a colocação pronominal intacta
    texto = re.sub(r'[.,!?;:()\[\]{}"\'°ªº/\\–—«»\u201c\u201d\u2018\u2019]', ' ', texto)
    texto = _restore_dots(texto)
    return re.sub(r'\s+', ' ', texto).strip()

def pre_processamento(texto, remover_stopwords=True, stop_words=portuguese_stopwords_list):
    if not isinstance(texto, str): return ""

    # 1. Limpeza Inicial
    texto = remove_noise(texto)
    # A linha abaixo foi removida/comentada para não separar os pronomes na ênclise/mesóclise
    # texto = re.sub(r'\b(\w+)-(se|me|te|lhe|no|nos|vos|lhes|lo|los)\b', r'\1 \2', texto, flags=re.IGNORECASE)
    texto = remove_whitespace(texto)

    if remover_stopwords:
        # Se for remover stopwords, o pipeline assume limpeza profunda (sem pontuação e lowercase)
        texto = remove_pontuacao(texto)
        texto = texto.lower()
        tokens = word_tokenize(texto, language='portuguese')
        filtered = [w for w in tokens if w not in stop_words and w.isalpha()]
        texto = " ".join(filtered)
    else:
        # Mantém a integridade original para análise gramatical ou visualização
        tokens = word_tokenize(texto, language='portuguese')
        texto = " ".join(tokens)
        texto = remove_whitespace(texto)

    return texto


In [ ]:
# Vamos verificar se 'e' está na lista padrão do NLTK
# print("'e' na lista do NLTK:", 'e' in portuguese_stopwords_list)

# # Podemos criar uma lista customizada adicionando mais palavras que você percebeu que não foram removidas
# custom_stopwords = set(portuguese_stopwords_list)
# custom_stopwords.update(['e', 'é', 'em', 'Em', 'o', 'a', 'os', 'as'])

# # E você pode atualizar a função pre_processamento para usar custom_stopwords no lugar de portuguese_stopwords_list
# print("Total de stopwords atualizadas:", len(custom_stopwords))

In [ ]:
# # Validação da correção: edge cases onde "e" escapava do filtro
# edge_cases = [
#     "tribunal e/ou substituição",         # e/ou (barra)
#     "5 anos e 10 meses",                  # e entre números
#     "semiaberto,e também",                # e colado com vírgula
#     "reclusão—e pagamento",               # e após travessão
#     "condenado 'e' pena",                 # e entre aspas simples
#     "Tribunal (e) origem",                # e entre parênteses
#     "O ponto final deveria estar aqui sem espaço.",  # regressão ponto
#     "Teste ponto. Ponto teste.",          # regressão ponto meio
# ]

# print("=" * 70)
# print("VALIDAÇÃO COM mantem_pontuacao='N'")
# print("=" * 70)
# all_ok = True
# for caso in edge_cases:
#     resultado = pre_processamento(caso, mantem_pontuacao='N', stop_words=portuguese_stopwords_list)
#     tem_e = 'e' in resultado.split()
#     marker = "⚠️ BUG!" if tem_e else "✓"
#     if tem_e:
#         all_ok = False
#     print(f"  {marker} |{caso}| → |{resultado}|")

# print()
# print("=" * 70)
# print("VALIDAÇÃO COM mantem_pontuacao='S'")
# print("=" * 70)
# for caso in edge_cases:
#     resultado = pre_processamento(caso, mantem_pontuacao='S', stop_words=portuguese_stopwords_list)
#     tem_espaco_ponto = ' .' in resultado
#     marker = "⚠️" if tem_espaco_ponto else "✓"
#     if tem_espaco_ponto:
#         all_ok = False
#     print(f"  {marker} |{caso}| → |{resultado}|")

# print(f"\n{'✅ Todos os testes passaram!' if all_ok else '❌ Há falhas!'}")

###Exemplo de uso das funções

In [ ]:
#Exemplo de uso da função
# texto = """  <br><hr> Trata-se , de   habeas   corpus substitutivo   <b> de </b>
# recurso          próprio , com pedido de liminar.Porém, tudo de bom.De bom.1.1253,00</br> """
# texto = remove_noise(texto)

# print(texto)

In [ ]:
# text = "583 dias-multa , incurso art. 33 , caput , c. c 40 , VI , Lei n . 11.343/2006.Em sede recursal , Tribunal origem negou provimento apelo defensivo"
# text = _consolidate_known_acronyms(text)
# print(text)

###Stopword removal


In [ ]:
# text = "<br>583 dias-multa , incurso art . 33 , <b>caput , </b>c. c 40 , VI , Lei n . 11.343/2006.Em sede recursal , Tribunal origem negou provimento apelo defensivo."

# text = remove_noise(text)
# print(f"After remove_noise: {text}")

# text = remove_whitespace(text)
# print(f"After remove_whitespace: {text}")

# text = _consolidate_known_acronyms(text)
# print(f"After remove_whitespace: {text}")


In [ ]:
#Exemplo de uso da função
# texto = """<br>O ministro presidente e relator (SEBASTIÃO REIS): Trata-se de habeas: corpus <b>substitutivo de recurso próprio</b>, com pedido de liminar, impetrado em favor de LUIS FELIPE SILVA DE ASSIS, em que se aponta com autoridade coatora o Tribunal de Justiça do Estado do Rio de Janeiro.Consta dos autos que o paciente foi condenado à pena de 5 anos e 10 meses de reclusão, em regime semiaberto, mais pagamento de 583 dias-multa, como incurso no art. 33,caput, c.c o 40, VI, da Lei n. 11.343/2006.Em sede recursal, o Tribunal de origem negou provimento ao apelo defensivo."""
# texto = pre_processamento(texto, 'S')

# print(texto)

In [ ]:
print ("version 1.08.0")

version 1.05.0
